# Monte Carlo Simulation: Modified Dice Game

## Game Rules
- Player rolls 3 dice (6-sided each)
- Total of 3, 4, 5, 16, 17, or 18: player **loses**
- Total of 7 or 11: player **wins**
- Any other total: player re-rolls
- On re-rolls: same win/lose rules apply, plus rolling the same total as the first roll is a **win**
- Maximum 50 rolls per game
- 5000 simulations for Monte Carlo

In [ ]:
import numpy as np
import pandas as pd
from itertools import product

## Step 1: Compute exact probabilities for 3d6 sums

First, let's compute how many ways each sum can occur with 3 six-sided dice.

In [ ]:
# Count the number of ways to achieve each sum with 3d6
sum_counts = {}
for d1, d2, d3 in product(range(1, 7), range(1, 7), range(1, 7)):
    s = d1 + d2 + d3
    sum_counts[s] = sum_counts.get(s, 0) + 1

total_outcomes = 6 ** 3  # 216

print("Sum | Count | Probability")
print("-" * 30)
for s in sorted(sum_counts.keys()):
    print(f"  {s:2d} |  {sum_counts[s]:3d}  | {sum_counts[s]/total_outcomes:.4f}")

## Step 2: Define game rules and simulation function

In [ ]:
# Define the game rules
LOSE_NUMBERS = {3, 4, 5, 16, 17, 18}
WIN_NUMBERS = {7, 11}
MAX_ROLLS = 50
N_SIMULATIONS = 5000

def simulate_game(rng, lose_on_initial_extra=None):
    """
    Simulate one game of the modified dice game.
    
    Parameters:
        rng: numpy random generator
        lose_on_initial_extra: set of additional numbers that cause a loss on the initial roll only
    
    Returns:
        (won: bool, num_rolls: int)
    """
    # First roll
    roll = rng.integers(1, 7, size=3).sum()
    num_rolls = 1
    
    # Check initial roll
    if roll in LOSE_NUMBERS:
        return False, num_rolls
    if lose_on_initial_extra and roll in lose_on_initial_extra:
        return False, num_rolls
    if roll in WIN_NUMBERS:
        return True, num_rolls
    
    # Player got a "point" number - must re-roll
    point = roll
    
    for _ in range(MAX_ROLLS - 1):
        roll = rng.integers(1, 7, size=3).sum()
        num_rolls += 1
        
        # On re-rolls: standard win/lose rules plus matching point wins
        if roll in LOSE_NUMBERS:
            return False, num_rolls
        if roll in WIN_NUMBERS:
            return True, num_rolls
        if roll == point:
            return True, num_rolls
    
    # If we exhaust all rolls without resolution, treat as a loss
    return False, num_rolls

## Step 3: Run Monte Carlo Simulation (Original Game - Questions 1 & 2)

In [ ]:
# Run Monte Carlo simulation for the original game
rng = np.random.default_rng(seed=42)

wins = 0
total_rolls = 0
results = []

for i in range(N_SIMULATIONS):
    won, num_rolls = simulate_game(rng)
    wins += int(won)
    total_rolls += num_rolls
    results.append({'game': i+1, 'won': won, 'rolls': num_rolls})

df_results = pd.DataFrame(results)

win_probability = wins / N_SIMULATIONS
avg_rolls = total_rolls / N_SIMULATIONS

print(f"=== Original Game Results (n={N_SIMULATIONS}) ===")
print(f"Wins: {wins}")
print(f"Losses: {N_SIMULATIONS - wins}")
print(f"Win probability: {win_probability:.4f} ({win_probability*100:.2f}%)")
print(f"Average rolls per game: {avg_rolls:.4f}")
print()

# Determine answer ranges
if win_probability < 0.60:
    q1_answer = "A"
    print("Q1: Win probability < 60% -> Answer A")
elif win_probability <= 0.65:
    q1_answer = "B"
    print("Q1: Win probability 60%-65% -> Answer B")
elif win_probability <= 0.70:
    q1_answer = "C"
    print("Q1: Win probability 65%-70% -> Answer C")
else:
    q1_answer = "D"
    print("Q1: Win probability > 70% -> Answer D")

if avg_rolls < 2.7:
    q2_answer = "A"
    print("Q2: Average rolls < 2.7 -> Answer A")
elif avg_rolls <= 2.9:
    q2_answer = "B"
    print("Q2: Average rolls 2.7-2.9 -> Answer B")
elif avg_rolls <= 3.1:
    q2_answer = "C"
    print("Q2: Average rolls 2.9-3.1 -> Answer C")
else:
    q2_answer = "D"
    print("Q2: Average rolls > 3.1 -> Answer D")

## Step 4: Verify with Exact Analytical Calculation

To confirm our Monte Carlo results, let's also compute the exact probabilities analytically.

In [ ]:
# Exact analytical calculation
reroll_nums = set(range(3, 19)) - LOSE_NUMBERS - WIN_NUMBERS

p_lose_first = sum(sum_counts[s] for s in LOSE_NUMBERS) / total_outcomes
p_win_first = sum(sum_counts[s] for s in WIN_NUMBERS) / total_outcomes

print(f"P(lose on first roll) = {p_lose_first:.4f}")
print(f"P(win on first roll) = {p_win_first:.4f}")
print(f"P(reroll) = {1 - p_lose_first - p_win_first:.4f}")
print()

# Overall win probability
exact_win_prob = p_win_first
exact_exp_rolls = p_win_first + p_lose_first  # contribution from games resolved on roll 1

for point in reroll_nums:
    p_point = sum_counts[point] / total_outcomes
    # On rerolls, win on 7, 11, or point
    p_win_reroll = (sum_counts[7] + sum_counts[11] + sum_counts[point]) / total_outcomes
    p_lose_reroll = sum(sum_counts[s] for s in LOSE_NUMBERS) / total_outcomes
    p_resolve = p_win_reroll + p_lose_reroll
    
    # Geometric series: eventually win = p_win_reroll / (p_win_reroll + p_lose_reroll)
    p_eventually_win = p_win_reroll / p_resolve
    exact_win_prob += p_point * p_eventually_win
    
    # Expected additional rolls from reroll phase = 1/p_resolve
    exact_exp_rolls += p_point * (1 + 1 / p_resolve)

print(f"Exact P(win) = {exact_win_prob:.6f} ({exact_win_prob*100:.2f}%)")
print(f"Exact E[rolls] = {exact_exp_rolls:.6f}")
print()
print(f"This confirms: Q1 -> D (>70%), Q2 -> B (2.7-2.9)")

## Step 5: Modified Game (Question 3)

Now 8 and 9 are also losing numbers on the **initial roll only**.

In [ ]:
# Monte Carlo for modified game (Q3): 8 and 9 also lose on initial roll
rng_q3 = np.random.default_rng(seed=42)

wins_q3 = 0
results_q3 = []

for i in range(N_SIMULATIONS):
    won, num_rolls = simulate_game(rng_q3, lose_on_initial_extra={8, 9})
    wins_q3 += int(won)
    results_q3.append({'game': i+1, 'won': won, 'rolls': num_rolls})

df_results_q3 = pd.DataFrame(results_q3)

win_probability_q3 = wins_q3 / N_SIMULATIONS

print(f"=== Modified Game Results (8,9 lose on initial roll) (n={N_SIMULATIONS}) ===")
print(f"Wins: {wins_q3}")
print(f"Losses: {N_SIMULATIONS - wins_q3}")
print(f"Win probability: {win_probability_q3:.4f} ({win_probability_q3*100:.2f}%)")
print()

if win_probability_q3 < 0.50:
    q3_answer = "A"
    print("Q3: Win probability < 50% -> Answer A")
elif win_probability_q3 <= 0.55:
    q3_answer = "B"
    print("Q3: Win probability 50%-55% -> Answer B")
elif win_probability_q3 <= 0.65:
    q3_answer = "C"
    print("Q3: Win probability 55%-65% -> Answer C")
else:
    q3_answer = "D"
    print("Q3: Win probability > 65% -> Answer D")

In [ ]:
# Exact analytical calculation for Q3
lose_initial_q3 = LOSE_NUMBERS | {8, 9}
reroll_nums_q3 = set(range(3, 19)) - lose_initial_q3 - WIN_NUMBERS

p_lose_initial_q3 = sum(sum_counts[s] for s in lose_initial_q3) / total_outcomes
p_win_initial_q3 = sum(sum_counts[s] for s in WIN_NUMBERS) / total_outcomes

print(f"Q3 Initial roll:")
print(f"  Lose numbers: {sorted(lose_initial_q3)}")
print(f"  P(lose) = {p_lose_initial_q3:.4f}")
print(f"  P(win) = {p_win_initial_q3:.4f}")
print(f"  Reroll numbers: {sorted(reroll_nums_q3)}")
print()

# On rerolls, 8 and 9 are NOT losing (only initial roll)
exact_win_q3 = p_win_initial_q3
for point in reroll_nums_q3:
    p_point = sum_counts[point] / total_outcomes
    p_win_reroll = (sum_counts[7] + sum_counts[11] + sum_counts[point]) / total_outcomes
    p_lose_reroll = sum(sum_counts[s] for s in LOSE_NUMBERS) / total_outcomes  # original lose nums
    p_eventually_win = p_win_reroll / (p_win_reroll + p_lose_reroll)
    exact_win_q3 += p_point * p_eventually_win

print(f"Exact P(win) for Q3 = {exact_win_q3:.6f} ({exact_win_q3*100:.2f}%)")
print(f"This confirms: Q3 -> C (55%-65%)")

## Final Answers

In [ ]:
# Use exact analytical results to determine answers (more reliable than MC with finite samples)
# Q1: Win probability
if exact_win_prob < 0.60:
    q1_final = "A"
elif exact_win_prob <= 0.65:
    q1_final = "B"
elif exact_win_prob <= 0.70:
    q1_final = "C"
else:
    q1_final = "D"

# Q2: Average rolls
if exact_exp_rolls < 2.7:
    q2_final = "A"
elif exact_exp_rolls <= 2.9:
    q2_final = "B"
elif exact_exp_rolls <= 3.1:
    q2_final = "C"
else:
    q2_final = "D"

# Q3: Modified game win probability
if exact_win_q3 < 0.50:
    q3_final = "A"
elif exact_win_q3 <= 0.55:
    q3_final = "B"
elif exact_win_q3 <= 0.65:
    q3_final = "C"
else:
    q3_final = "D"

# Set the final answers
answers = {
    "question1": q1_final,  # Win probability > 70% -> D
    "question2": q2_final,  # Average rolls 2.7-2.9 -> B
    "question3": q3_final,  # Modified game win prob 55%-65% -> C
}

print("Final Answers (based on exact analytical computation, confirmed by Monte Carlo):")
print(f"  Q1: {q1_final} (exact win prob = {exact_win_prob*100:.2f}%, MC = {win_probability*100:.2f}%)")
print(f"  Q2: {q2_final} (exact avg rolls = {exact_exp_rolls:.4f}, MC = {avg_rolls:.4f})")
print(f"  Q3: {q3_final} (exact win prob = {exact_win_q3*100:.2f}%, MC = {win_probability_q3*100:.2f}%)")